In [1]:
!pip install scikit-learn pandas matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import json
import joblib
import os

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
os.makedirs("outputs", exist_ok=True)


In [5]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Dataset Shape:", X.shape)
X.head()


Dataset Shape: (569, 30)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (455, 30)
Test: (114, 30)


In [9]:
models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "RandomForest": Pipeline([
        ("model", RandomForestClassifier(random_state=42))
    ])
}

param_grids = {
    "SVM": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    },

    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_split": [2, 5]
    }
}


In [11]:
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, pred)

    results.append({
        "Model": name,
        "Type": "Default",
        "Accuracy": acc
    })

print("Default models trained.")


Default models trained.


In [12]:
best_models = {}

for name in models:
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        models[name],
        param_grids[name],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    pred = grid.predict(X_test)
    acc = accuracy_score(y_test, pred)

    results.append({
        "Model": name,
        "Type": "Tuned",
        "Accuracy": acc
    })

    # Save best params
    with open(f"outputs/{name}_best_params.json", "w") as f:
        json.dump(grid.best_params_, f, indent=4)

    print("Best Params:", grid.best_params_)



Tuning SVM ...
Best Params: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}

Tuning RandomForest ...
Best Params: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [14]:
results_df = pd.DataFrame(results)
results_df.to_csv("outputs/performance_comparison.csv", index=False)

results_df


,Model,Type,Accuracy
0,SVM,Default,0.982456
1,RandomForest,Default,0.956140
2,SVM,Tuned,0.982456
3,RandomForest,Tuned,0.956140


In [16]:
for name, model in best_models.items():
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title(f"{name} Confusion Matrix")

    plt.savefig(f"outputs/{name}_confusion_matrix.png")
    plt.close()

print("Confusion matrices saved.")


Confusion matrices saved.


In [18]:
for name, model in best_models.items():
    pred = model.predict(X_test)
    report = classification_report(y_test, pred)

    with open(f"outputs/{name}_classification_report.txt", "w") as f:
        f.write(report)

print("Reports saved.")


Reports saved.


In [20]:
for name, model in best_models.items():
    joblib.dump(model, f"outputs/{name}_best_model.pkl")

print("Models saved.")


Models saved.


In [22]:
readme = """
# Hyperparameter Tuning using GridSearchCV

## Dataset
Breast Cancer Dataset from sklearn

## Models Tuned
- Support Vector Machine
- Random Forest

## Method
- Train/Test Split
- GridSearchCV with 5-fold CV
- Compared default vs tuned models

## Outputs Saved
- Best Parameters JSON
- Performance CSV
- Confusion Matrix Images
- Classification Reports
- Saved Models

## Unique Approach
Dual-model tuning and automated artifact saving pipeline.
"""

with open("README.md", "w") as f:
    f.write(readme)

print("README.md created.")


README.md created.


In [24]:
report = f"""
# Internship Task 16 Report — GridSearchCV

## Objective
Perform hyperparameter tuning and compare model performance.

## Models Used
SVM and Random Forest

## Cross Validation
5-fold cross validation used inside GridSearchCV.

## Results
{results_df.to_markdown(index=False)}

## Best Parameters
Check outputs folder JSON files.

## Conclusion
Hyperparameter tuning improved model performance compared to default settings.
"""

with open("REPORT.md", "w") as f:
    f.write(report)

print("REPORT.md created.")


REPORT.md created.


In [26]:
!zip -r outputs.zip outputs README.md REPORT.md


updating: outputs/ (stored 0%)
updating: outputs/SVM_confusion_matrix.png (deflated 20%)
updating: outputs/SVM_classification_report.txt (deflated 66%)
updating: outputs/performance_comparison.csv (deflated 46%)
updating: outputs/RandomForest_classification_report.txt (deflated 62%)
updating: outputs/SVM_best_params.json (deflated 31%)
updating: outputs/RandomForest_confusion_matrix.png (deflated 18%)
updating: outputs/RandomForest_best_model.pkl (deflated 81%)
updating: outputs/SVM_best_model.pkl (deflated 16%)
updating: outputs/RandomForest_best_params.json (deflated 27%)
updating: README.md (deflated 36%)
updating: REPORT.md (deflated 48%)
